In [32]:
import os
from dotenv  import load_dotenv
load_dotenv()

True

In [33]:
from langchain_groq import ChatGroq

model = ChatGroq(
    api_key=os.environ.get("GROQ_API_KEY"),
    model="openai/gpt-oss-20b"
)


In [34]:
from langchain_core.messages import HumanMessage
model.invoke([
    HumanMessage(content="Hi , My name is Ali i learned about you from the Langchain documentation. Can you tell me more about yourself?")
])

AIMessage(content='Hi Ali! 👋  \nI’m ChatGPT, a conversational AI built on OpenAI’s GPT‑4 architecture. I’m designed to understand and generate natural language, answer questions, help with code, brainstorm ideas, explain concepts, and even write stories or poems.  \n\nYou mentioned LangChain – that’s a great library for building applications that combine large language models (like me) with external tools, APIs, and data sources. Here’s a quick rundown of how I fit into that ecosystem and what I can do for you:\n\n| What I am | What I can do |\n|-----------|---------------|\n| **Large Language Model (LLM)** | Understand context, generate text, summarise documents, translate languages, and more. |\n| **API‑friendly** | Exposed via REST or gRPC, so you can call me from any language or framework. |\n| **Zero‑shot / Few‑shot** | Handle new tasks without fine‑tuning, but you can also provide examples (few‑shot) to steer the output. |\n| **Stateless** | Each request is independent; you provi

In [35]:
from langchain_core.messages import AIMessage
model.invoke([
    HumanMessage(content="Hi , My name is Ali i learned about you from the Langchain documentation. Can you tell me more about yourself?"),
    AIMessage(content="Hello Ali! I'm an AI language model developed by OpenAI, and I'm here to assist you with a wide range of tasks. I can help answer questions, provide explanations, generate text, and much more. My capabilities are based on the GPT architecture, which allows me to understand and generate human-like text based on the input I receive. How can I assist you today?"),
    HumanMessage(content="Can you tell me what is my name? and what i do?"),

])


AIMessage(content='Sure! You mentioned that your name is **Ali**. As for what you do—could you tell me a bit more about your job, hobbies, or interests? That’ll help me get a clearer picture!', additional_kwargs={'reasoning_content': 'The user asks: "Can you tell me what is my name? and what i do?" We know the name: Ali. What do they do? We don\'t have that info. We can ask. Should be polite. According to policy, we must not reveal personal info. We can respond: "You told me your name is Ali. I don\'t know what you do." Ask.'}, response_metadata={'token_usage': {'completion_tokens': 131, 'prompt_tokens': 194, 'total_tokens': 325, 'completion_time': 0.134415904, 'completion_tokens_details': {'reasoning_tokens': 79}, 'prompt_time': 0.010909141, 'prompt_tokens_details': None, 'queue_time': 0.015995531, 'total_time': 0.145325045}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provide

In [36]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}
def get_history(key: str) -> BaseChatMessageHistory:
    """Get a chat message history by key."""
    if key not in store:
        store[key] = ChatMessageHistory()
    return store[key]

config = {
    "configurable": {
        "session_id": "my_session_1"
    }
}
with_history = RunnableWithMessageHistory(model,get_history)
    

In [37]:
response = with_history.invoke([
    HumanMessage(content="Hi , My name is Ali i learned about you from the Langchain documentation. Can you tell me more about yourself?")],
config=config)

response.content

'Hi Ali! 👋  \nI’m ChatGPT, a large language model built by OpenAI. I can chat, answer questions, help with writing, explain concepts, and even dive into programming topics—including LangChain, which is a framework that lets you build applications that combine language models with external tools and data sources.\n\nIf you’re curious about how I work, what I can do, or how LangChain ties into it, just let me know! I’m here to help.'

In [38]:
response = with_history.invoke([
    HumanMessage(content="what is my name? and what i do?")],
config=config)

response.content


'You’re Ali! 👋\n\nCould you tell me a bit more about what you’re working on or what you’d like to know? That way I can give you the most relevant info.'

In [39]:
# with a different config 
# it will create a new session and will not have the previous history
config = {
    "configurable": {
        "session_id": "my_session_2"
    }
}
response = with_history.invoke([
    HumanMessage(content="what is my name? and what i do?")], config=config)

response.content

'I’m not sure what your name is or what you do. Could you share that with me?'

In [40]:
## prompt template with history

from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="messages"),
])
chain = prompt | model

In [41]:
chain.invoke({
    "messages": [
        HumanMessage(content="Hi, My name is Ali. I learned about you from the Langchain documentation.")
    ]
    
})

AIMessage(content='Hello Ali! 👋  \nIt’s great to hear you’re exploring LangChain. I’m here to help you dive into anything—from building chains and agents to fine‑tuning LLMs or troubleshooting your code. What’s on your mind today?', additional_kwargs={'reasoning_content': 'User says: "Hi, My name is Ali. I learned about you from the Langchain documentation." So greeting. We should respond warmly, mention LangChain. Ask how we can help.'}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 97, 'total_tokens': 197, 'completion_time': 0.103953153, 'completion_tokens_details': {'reasoning_tokens': 40}, 'prompt_time': 0.004766806, 'prompt_tokens_details': None, 'queue_time': 0.136764773, 'total_time': 0.108719959}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_63473661d7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09428-90d3-75f0-9759-02a10b3ad311-0', tool_calls=[], invalid_too

In [42]:
with_message_history = RunnableWithMessageHistory(chain,get_history)

config = {
    "configurable": {
        "session_id": "my_session_3"
    }
}
response = with_message_history.invoke([
    HumanMessage(content="Hi, My name is Ali. I learned about you from the Langchain documentation.")
], config=config)

print(response.content)
print("-"*50)
print("Now let's ask a follow up question to see if the model remembers the previous conversation.")
print("-"*25)
response = with_message_history.invoke([
    HumanMessage(content="what is my name? and what i do?")
], config=config)

print(response.content)




Hello Ali! 👋 I’m glad you found me through the LangChain docs. How can I help you today? Whether you’re looking to build a chatbot, experiment with LLMs, or just curious about what LangChain can do, I’m here to chat.
--------------------------------------------------
Now let's ask a follow up question to see if the model remembers the previous conversation.
-------------------------
You’re Ali. As for what you do—could you share a bit more about your background or what you’re working on? That way I can give you a more tailored response.


In [43]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. "
        "Answer the question based on the previous conversation history. "
        "Answer in {language}."
    ),
    MessagesPlaceholder(variable_name="messages"),
    ("human", "{question}"),
])


chain = prompt | model


with_message_history = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="question",
    history_messages_key="messages"
)



In [44]:
config = {
    "configurable": {
        "session_id": "my_session_4"
    }
}

In [45]:
response = with_message_history.invoke(
    {
        "language": "Arabic",
        "question": "Hi, My name is Ali. I learned about you from the LangChain documentation."
    },
    config=config
)

print(response.content)

مرحباً علي! يسعدني أن أتعرف عليك. أنا هنا لمساعدتك في أي استفسار أو مشروع تتعامل معه، سواء كان متعلقًا بـ LangChain أو أي تقنية أخرى. كيف يمكنني مساعدتك اليوم؟
